# **DATA CLEANING PIPELINE**

In [ ]:
import pandas as pd
import numpy as np
import os
import gc

# paths


capstone_path = "/content/drive/MyDrive/Capstone_Project/Data"
lending_path = "/content/drive/MyDrive/LendingClub"

macro_file = os.path.join(capstone_path, "macro_fred.csv")
fdic_file = os.path.join(capstone_path, "fdic_financials.csv")
loan_file = os.path.join(lending_path, "accepted_2007_to_2018Q4.csv.gz")

output_path = os.path.join(capstone_path, "cleaned")
os.makedirs(output_path, exist_ok=True)

print("Output folder:", output_path)

Output folder: /content/drive/MyDrive/Capstone_Project/Data/cleaned


In [ ]:
# Helper functions


def reduce_memory(df):
    for col in df.columns:
        col_type = df[col].dtype

        if col_type != "object":
            c_min = df[col].min()
            c_max = df[col].max()

            if str(col_type).startswith("int"):
                if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)

            elif str(col_type).startswith("float"):
                df[col] = df[col].astype(np.float32)

        else:
            if df[col].nunique(dropna=True) / len(df) < 0.5:
                df[col] = df[col].astype("category")

    return df


def clean_percent_column(series):
    return (
        series.astype(str)
        .str.replace("%", "", regex=False)
        .str.strip()
        .replace("nan", np.nan)
        .astype(float)
    )


def clean_emp_length(series):
    return (
        series.astype(str)
        .str.replace("years", "", regex=False)
        .str.replace("year", "", regex=False)
        .str.replace("+", "", regex=False)
        .str.replace("< 1", "0", regex=False)
        .str.strip()
        .replace("nan", np.nan)
        .astype(float)
    )

# Load LendingClub data


loan_cols = [
    "id",
    "loan_amnt",
    "funded_amnt",
    "term",
    "int_rate",
    "installment",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "issue_d",
    "loan_status",
    "purpose",
    "dti",
    "delinq_2yrs",
    "earliest_cr_line",
    "fico_range_low",
    "fico_range_high",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "mort_acc",
    "pub_rec_bankruptcies"
]

loans = pd.read_csv(
    loan_file,
    usecols=lambda c: c in loan_cols,
    low_memory=False
)

print("Raw loans shape:", loans.shape)
loans.head()

Raw loans shape: (2260701, 28)


,id,loan_amnt,funded_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,...,fico_range_low,fico_range_high,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,mort_acc,pub_rec_bankruptcies
0,68407277,3600.0,3600.0,36 months,13.99,123.03,C,C4,10+ years,MORTGAGE,...,675.0,679.0,1.0,7.0,0.0,2765.0,29.7,13.0,1.0,0.0
1,68355089,24700.0,24700.0,36 months,11.99,820.28,C,C1,10+ years,MORTGAGE,...,715.0,719.0,4.0,22.0,0.0,21470.0,19.2,38.0,4.0,0.0
2,68341763,20000.0,20000.0,60 months,10.78,432.66,B,B4,10+ years,MORTGAGE,...,695.0,699.0,0.0,6.0,0.0,7869.0,56.2,18.0,5.0,0.0
3,66310712,35000.0,35000.0,60 months,14.85,829.90,C,C5,10+ years,MORTGAGE,...,785.0,789.0,0.0,13.0,0.0,7802.0,11.6,17.0,1.0,0.0
4,68476807,10400.0,10400.0,60 months,22.45,289.91,F,F1,3 years,MORTGAGE,...,695.0,699.0,3.0,12.0,0.0,21929.0,64.5,35.0,6.0,0.0


In [ ]:
# Clean LendingClub data


loans = loans.drop_duplicates()

# Clean interest rate and revolving utilization
loans["int_rate"] = clean_percent_column(loans["int_rate"])
loans["revol_util"] = clean_percent_column(loans["revol_util"])

# Clean employment length
loans["emp_length"] = clean_emp_length(loans["emp_length"])

# Clean term
loans["term_months"] = (
    loans["term"]
    .astype(str)
    .str.extract(r"(\d+)")
    .astype(float)
)

# Dates
loans["issue_d"] = pd.to_datetime(loans["issue_d"], format="%b-%Y", errors="coerce")
loans["earliest_cr_line"] = pd.to_datetime(
    loans["earliest_cr_line"], format="%b-%Y", errors="coerce"
)

# Credit history length
loans["credit_history_months"] = (
    (loans["issue_d"].dt.year - loans["earliest_cr_line"].dt.year) * 12
    + (loans["issue_d"].dt.month - loans["earliest_cr_line"].dt.month)
)

# Average FICO
loans["fico_avg"] = (
    loans["fico_range_low"] + loans["fico_range_high"]
) / 2

# Create target variable
default_statuses = [
    "Charged Off",
    "Default",
    "Does not meet the credit policy. Status:Charged Off"
]

good_statuses = [
    "Fully Paid",
    "Does not meet the credit policy. Status:Fully Paid"
]

loans = loans[loans["loan_status"].isin(default_statuses + good_statuses)].copy()

loans["default_flag"] = loans["loan_status"].isin(default_statuses).astype(int)

# Remove columns not needed after transformation
loans = loans.drop(
    columns=[
        "term",
        "earliest_cr_line",
        "fico_range_low",
        "fico_range_high"
    ],
    errors="ignore"
)

print("Cleaned loans shape:", loans.shape)
print(loans["default_flag"].value_counts(normalize=True))
loans.head()

# Handle missing values


numeric_cols = loans.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns
categorical_cols = loans.select_dtypes(include=["object", "category"]).columns

for col in numeric_cols:
    loans[col] = loans[col].fillna(loans[col].median())

for col in categorical_cols:
    loans[col] = loans[col].astype(str).fillna("Unknown")

loans = reduce_memory(loans)

print(loans.info())

Cleaned loans shape: (1348099, 28)
default_flag
0    0.800193
1    0.199807
Name: proportion, dtype: float64
<class 'pandas.core.frame.DataFrame'>
Index: 1348099 entries, 0 to 2260697
Data columns (total 28 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   id                     1348099 non-null  object        
 1   loan_amnt              1348099 non-null  float32       
 2   funded_amnt            1348099 non-null  float32       
 3   int_rate               1348099 non-null  float32       
 4   installment            1348099 non-null  float32       
 5   grade                  1348099 non-null  category      
 6   sub_grade              1348099 non-null  category      
 7   emp_length             1348099 non-null  float32       
 8   home_ownership         1348099 non-null  category      
 9   annual_inc             1348099 non-null  float32       
 10  verification_status    1348099 non-null  categ

In [ ]:

# Load and clean macro data


macro = pd.read_csv(macro_file)

print("Raw macro shape:", macro.shape)
print(macro.head())

# Rename first column to date
macro = macro.rename(columns={"Unnamed: 0": "date"})

# Convert to datetime
macro["date"] = pd.to_datetime(macro["date"])

# Remove duplicates
macro = macro.drop_duplicates().sort_values("date")

# Ensure numeric columns
macro_cols = ["GDP", "UNRATE", "CPI", "FEDFUNDS", "HPI"]

for col in macro_cols:
    macro[col] = pd.to_numeric(macro[col], errors="coerce")

# Fill missing values
macro[macro_cols] = macro[macro_cols].ffill().bfill()

# Time features
macro["year"] = macro["date"].dt.year
macro["month"] = macro["date"].dt.month
macro["quarter"] = macro["date"].dt.to_period("Q").astype(str)

# Reduce memory
macro = reduce_memory(macro)

print(macro.info())
macro.head()

Raw macro shape: (60, 6)
   Unnamed: 0        GDP  UNRATE      CPI  FEDFUNDS      HPI
0  2010-01-01  14764.610     9.8  217.488      0.11  147.396
1  2010-04-01  14980.193     9.9  217.403      0.20  146.398
2  2010-07-01  15141.607     9.4  217.605      0.18  144.984
3  2010-10-01  15309.474     9.4  219.035      0.19  142.523
4  2011-01-01  15351.448     9.1  221.187      0.17  141.511
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   date      60 non-null     datetime64[ns]
 1   GDP       60 non-null     float32       
 2   UNRATE    60 non-null     float32       
 3   CPI       60 non-null     float32       
 4   FEDFUNDS  60 non-null     float32       
 5   HPI       60 non-null     float32       
 6   year      60 non-null     int16         
 7   month     60 non-null     int8          
 8   quarter   60 non-null     object        
d

,date,GDP,UNRATE,CPI,FEDFUNDS,HPI,year,month,quarter
0,2010-01-01,14764.610352,9.8,217.488007,0.11,147.395996,2010,1,2010Q1
1,2010-04-01,14980.193359,9.9,217.403000,0.20,146.397995,2010,4,2010Q2
2,2010-07-01,15141.607422,9.4,217.604996,0.18,144.983994,2010,7,2010Q3
3,2010-10-01,15309.473633,9.4,219.035004,0.19,142.522995,2010,10,2010Q4
4,2011-01-01,15351.448242,9.1,221.186996,0.17,141.511002,2011,1,2011Q1


# **Optional Data (not used yet)**

In [ ]:
# Load and clean FDIC data


fdic = pd.read_csv(fdic_file, low_memory=False)

print("Raw FDIC shape:", fdic.shape)
print(fdic.head())

fdic = fdic.drop_duplicates()

# Standardize column names
fdic.columns = (
    fdic.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Try to parse date columns
for col in fdic.columns:
    if "date" in col or "quarter" in col:
        fdic[col] = pd.to_datetime(fdic[col], errors="ignore")

# Convert numeric-looking columns
for col in fdic.columns:
    if fdic[col].dtype == "object":
        converted = pd.to_numeric(fdic[col], errors="coerce")
        if converted.notna().mean() > 0.6:
            fdic[col] = converted

numeric_fdic_cols = fdic.select_dtypes(include=["int64", "float64", "float32", "int32"]).columns
categorical_fdic_cols = fdic.select_dtypes(include=["object", "category"]).columns

for col in numeric_fdic_cols:
    fdic[col] = fdic[col].fillna(fdic[col].median())

for col in categorical_fdic_cols:
    fdic[col] = fdic[col].astype(str).fillna("Unknown")

fdic = reduce_memory(fdic)

print("Cleaned FDIC shape:", fdic.shape)
fdic.head()

Raw FDIC shape: (10000, 8)
     REPDTE   ASSET   CERT  LNLSGR     EQ     DEP                 NAME  \
0  20100331  282913  10002  126266  30923  222888  PROGRESSIVE BANK NA   
1  20100630  274828  10002  121959  31970  221148  PROGRESSIVE BANK NA   
2  20100930  282396  10002  123462  32520  229859  PROGRESSIVE BANK NA   
3  20101231  277352  10002  121367  30845  228612  PROGRESSIVE BANK NA   
4  20110331  283551  10002  117324  31295  234217  PROGRESSIVE BANK NA   

               ID  
0  10002_20100331  
1  10002_20100630  
2  10002_20100930  
3  10002_20101231  
4  10002_20110331  
Cleaned FDIC shape: (10000, 8)


,repdte,asset,cert,lnlsgr,eq,dep,name,id
0,20100331,282913,10002,126266,30923,222888,PROGRESSIVE BANK NA,10002_20100331
1,20100630,274828,10002,121959,31970,221148,PROGRESSIVE BANK NA,10002_20100630
2,20100930,282396,10002,123462,32520,229859,PROGRESSIVE BANK NA,10002_20100930
3,20101231,277352,10002,121367,30845,228612,PROGRESSIVE BANK NA,10002_20101231
4,20110331,283551,10002,117324,31295,234217,PROGRESSIVE BANK NA,10002_20110331


In [ ]:
# Add loan time keys for macro merge later

loans["year"] = loans["issue_d"].dt.year
loans["month"] = loans["issue_d"].dt.month
loans["quarter"] = loans["issue_d"].dt.to_period("Q").astype(str)

print(loans[["issue_d", "year", "month", "quarter"]].head())

     issue_d  year  month quarter
0 2015-12-01  2015     12  2015Q4
1 2015-12-01  2015     12  2015Q4
2 2015-12-01  2015     12  2015Q4
4 2015-12-01  2015     12  2015Q4
5 2015-12-01  2015     12  2015Q4


# **Missing value report**

In [ ]:



missing_report = pd.DataFrame({
    "missing_count": loans.isna().sum(),
    "missing_percent": loans.isna().mean() * 100
}).sort_values("missing_percent", ascending=False)

missing_report.head(30)

,missing_count,missing_percent
id,0,0.0
loan_amnt,0,0.0
funded_amnt,0,0.0
int_rate,0,0.0
installment,0,0.0
grade,0,0.0
sub_grade,0,0.0
emp_length,0,0.0
home_ownership,0,0.0
annual_inc,0,0.0


In [ ]:
# Remove impossible values


print("Before impossible-value cleaning:", loans.shape)

loans = loans[
    (loans["loan_amnt"] > 0) &
    (loans["funded_amnt"] > 0) &
    (loans["annual_inc"] > 0) &
    (loans["installment"] > 0) &
    (loans["dti"] >= 0) &
    (loans["int_rate"] >= 0) &
    (loans["fico_avg"].between(300, 850)) &
    (loans["credit_history_months"] >= 0)
].copy()

print("After impossible-value cleaning:", loans.shape)

Before impossible-value cleaning: (1348099, 31)
After impossible-value cleaning: (1347736, 31)


# **Outlier detection report**

In [ ]:
outlier_cols = [
    "loan_amnt",
    "funded_amnt",
    "annual_inc",
    "installment",
    "dti",
    "int_rate",
    "revol_bal",
    "revol_util",
    "credit_history_months",
    "fico_avg"
]

outlier_report = []

for col in outlier_cols:
    if col in loans.columns:
        q1 = loans[col].quantile(0.25)
        q3 = loans[col].quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        count = ((loans[col] < lower) | (loans[col] > upper)).sum()
        percent = count / len(loans) * 100

        outlier_report.append([
            col,
            count,
            percent,
            lower,
            upper
        ])

outlier_report = pd.DataFrame(
    outlier_report,
    columns=[
        "feature",
        "outlier_count",
        "outlier_percent",
        "iqr_lower",
        "iqr_upper"
    ]
)

outlier_report = outlier_report.sort_values(
    "outlier_percent",
    ascending=False
)

outlier_report

,feature,outlier_count,outlier_percent,iqr_lower,iqr_upper
6,revol_bal,80037,5.938626,-14794.500000,40489.500000
2,annual_inc,66033,4.899550,-20600.000000,156360.000000
9,fico_avg,46497,3.450008,612.000000,772.000000
8,credit_history_months,42800,3.175696,-25.000000,399.000000
3,installment,42167,3.128728,-249.654945,1078.144920
5,int_rate,24957,1.851772,0.390000,25.349999
0,loan_amnt,7135,0.529406,-10062.500000,38037.500000
1,funded_amnt,7130,0.529035,-10125.000000,38075.000000
4,dti,5489,0.407276,-6.599999,42.439998
7,revol_util,72,0.005342,-22.299995,126.499992


# **Winsorize extreme values**

In [ ]:
# Winsorize financial outliers


winsor_cols = [
    "loan_amnt",
    "funded_amnt",
    "annual_inc",
    "installment",
    "dti",
    "int_rate",
    "revol_bal",
    "revol_util"
]

for col in winsor_cols:
    if col in loans.columns:
        lower = loans[col].quantile(0.01)
        upper = loans[col].quantile(0.99)
        loans[col] = loans[col].clip(lower, upper)

print("Winsorization completed.")

Winsorization completed.


# **Final null check**

In [ ]:
numeric_cols = loans.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns
categorical_cols = loans.select_dtypes(include=["object", "category"]).columns

for col in numeric_cols:
    loans[col] = loans[col].fillna(loans[col].median())

for col in categorical_cols:
    loans[col] = loans[col].astype(str).fillna("Unknown")

print("Total missing values:", loans.isna().sum().sum())

Total missing values: 0


# **Merge macro data**

In [ ]:
loans_macro = loans.merge(
    macro.drop(columns=["date", "month"], errors="ignore"),
    on=["year", "quarter"],
    how="left"
)

print("Loans after macro merge:", loans_macro.shape)
print("Missing after macro merge:", loans_macro.isna().sum().sum())

loans_macro.head()

Loans after macro merge: (1347736, 36)
Missing after macro merge: 41385


,id,loan_amnt,funded_amnt,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,...,fico_avg,default_flag,year,month,quarter,GDP,UNRATE,CPI,FEDFUNDS,HPI
0,68407277,3600.0,3600.0,13.990000,123.029999,C,C4,10.0,MORTGAGE,55000.0,...,677.0,0,2015,12,2015Q4,18435.136719,5.0,237.733002,0.12,174.764008
1,68355089,24700.0,24700.0,11.990000,820.280029,C,C1,10.0,MORTGAGE,65000.0,...,717.0,0,2015,12,2015Q4,18435.136719,5.0,237.733002,0.12,174.764008
2,68341763,20000.0,20000.0,10.780000,432.660004,B,B4,10.0,MORTGAGE,63000.0,...,697.0,0,2015,12,2015Q4,18435.136719,5.0,237.733002,0.12,174.764008
3,68476807,10400.0,10400.0,22.450001,289.910004,F,F1,3.0,MORTGAGE,104433.0,...,697.0,0,2015,12,2015Q4,18435.136719,5.0,237.733002,0.12,174.764008
4,68426831,11950.0,11950.0,13.440000,405.179993,C,C3,4.0,RENT,34000.0,...,692.0,0,2015,12,2015Q4,18435.136719,5.0,237.733002,0.12,174.764008


In [ ]:
# Diagnose macro merge missing values


print("Macro date range:")
print("Start:", macro["date"].min())
print("End  :", macro["date"].max())

print("\nMissing values after macro merge:")
print(loans_macro.isna().sum().sort_values(ascending=False).head(20))

Macro date range:
Start: 2010-01-01 00:00:00
End  : 2024-10-01 00:00:00

Missing values after macro merge:
GDP                    8277
CPI                    8277
FEDFUNDS               8277
HPI                    8277
UNRATE                 8277
funded_amnt               0
loan_amnt                 0
id                        0
int_rate                  0
installment               0
sub_grade                 0
grade                     0
loan_status               0
purpose                   0
dti                       0
emp_length                0
home_ownership            0
annual_inc                0
verification_status       0
issue_d                   0
dtype: int64


In [ ]:
# Check unmatched loan quarters


macro_vars = ["GDP", "UNRATE", "CPI", "FEDFUNDS", "HPI"]

missing_macro_rows = loans_macro[
    loans_macro[macro_vars].isna().any(axis=1)
].copy()

print("Rows with missing macro values:", missing_macro_rows.shape)

missing_macro_rows[
    ["issue_d", "year", "quarter"]
].drop_duplicates().sort_values("issue_d").head(20)

Rows with missing macro values: (8277, 36)


,issue_d,year,quarter
957710,2007-06-01,2007,2007Q2
957675,2007-07-01,2007,2007Q3
957646,2007-08-01,2007,2007Q3
957629,2007-09-01,2007,2007Q3
957582,2007-10-01,2007,2007Q4
957533,2007-11-01,2007,2007Q4
957458,2007-12-01,2007,2007Q4
957252,2008-01-01,2008,2008Q1
957098,2008-02-01,2008,2008Q1
956833,2008-03-01,2008,2008Q1


In [ ]:
# Final safety fill if any missing values remain


num_cols = loans_macro.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns
cat_cols = loans_macro.select_dtypes(include=["object", "category"]).columns

for col in num_cols:
    loans_macro[col] = loans_macro[col].fillna(loans_macro[col].median())

for col in cat_cols:
    loans_macro[col] = loans_macro[col].astype(str).fillna("Unknown")

print("Final missing values:", loans_macro.isna().sum().sum())
print("Final shape:", loans_macro.shape)

Final missing values: 0
Final shape: (1347736, 36)


# **Save Final Dataset**

In [ ]:
# Save Final Dataset


loans_macro = reduce_memory(loans_macro)

csv_file = os.path.join(
    output_path,
    "loans_macro_cleaned_final.csv"
)

parquet_file = os.path.join(
    output_path,
    "loans_macro_cleaned_final.parquet"
)

loans_macro.to_csv(csv_file, index=False)
loans_macro.to_parquet(parquet_file, index=False)

print("Saved successfully!")
print(csv_file)
print(parquet_file)
print("Final Shape:", loans_macro.shape)

Saved successfully!
/content/drive/MyDrive/Capstone_Project/Data/cleaned/loans_macro_cleaned_final.csv
/content/drive/MyDrive/Capstone_Project/Data/cleaned/loans_macro_cleaned_final.parquet
Final Shape: (1347736, 36)


# **Import libraries**

In [ ]:
import pandas as pd
import numpy as np
import os

# **Define the path**

In [ ]:
# Path to cleaned data

cleaned_path = "/content/drive/MyDrive/Capstone_Project/Data/cleaned"

# **Load the cleaned dataset**

In [ ]:
df = pd.read_parquet(
    os.path.join(cleaned_path, "loans_macro_cleaned_final.parquet")
)

print(df.shape)

df.head()

(1347736, 36)


,id,loan_amnt,funded_amnt,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,...,fico_avg,default_flag,year,month,quarter,GDP,UNRATE,CPI,FEDFUNDS,HPI
0,68407277,3600.0,3600.0,13.990000,123.029999,C,C4,10.0,MORTGAGE,55000.0,...,677.0,0,2015,12,2015Q4,18435.136719,5.0,237.733002,0.12,174.764008
1,68355089,24700.0,24700.0,11.990000,820.280029,C,C1,10.0,MORTGAGE,65000.0,...,717.0,0,2015,12,2015Q4,18435.136719,5.0,237.733002,0.12,174.764008
2,68341763,20000.0,20000.0,10.780000,432.660004,B,B4,10.0,MORTGAGE,63000.0,...,697.0,0,2015,12,2015Q4,18435.136719,5.0,237.733002,0.12,174.764008
3,68476807,10400.0,10400.0,22.450001,289.910004,F,F1,3.0,MORTGAGE,104433.0,...,697.0,0,2015,12,2015Q4,18435.136719,5.0,237.733002,0.12,174.764008
4,68426831,11950.0,11950.0,13.440000,405.179993,C,C3,4.0,RENT,34000.0,...,692.0,0,2015,12,2015Q4,18435.136719,5.0,237.733002,0.12,174.764008


# **Verify the dataset**

In [ ]:
print("="*50)
print("Dataset Information")
print("="*50)

print(df.info())

print("\nShape:", df.shape)

print("\nMissing Values:")
print(df.isnull().sum().sum())

print("\nTarget Distribution")
print(df["default_flag"].value_counts())

print("\nTarget Percentage")
print(df["default_flag"].value_counts(normalize=True) * 100)

Dataset Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1347736 entries, 0 to 1347735
Data columns (total 36 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   id                     1347736 non-null  object        
 1   loan_amnt              1347736 non-null  float32       
 2   funded_amnt            1347736 non-null  float32       
 3   int_rate               1347736 non-null  float32       
 4   installment            1347736 non-null  float32       
 5   grade                  1347736 non-null  category      
 6   sub_grade              1347736 non-null  category      
 7   emp_length             1347736 non-null  float32       
 8   home_ownership         1347736 non-null  category      
 9   annual_inc             1347736 non-null  float32       
 10  verification_status    1347736 non-null  category      
 11  issue_d                1347736 non-null  datetime64[ns]
 12  loan_sta